Copyright Matlantis Corp. as contributors to Matlantis contrib project

# RC≈12 付近の追加アンブレラサンプリング（バネ定数を強化）

弱いバネ定数（`KAPPA=2.5 eV`, ウインドウ間隔 0.2 Å）では拘束中心から系が滑り落ちてしまい、隣接ウィンドウ間のヒストグラムの重なりが不十分になることがあります。

このノートブックでは、12 Å 付近だけを対象に

- バネ定数 `KAPPA` を強く （デフォルト `2.5 → 10.0 eV`）
- ウインドウ間隔を細かく （0.2 → 0.1 Å）

して追加のアンブレラサンプリングを行います。

次の `08_mbar_free_energy_with_extra_ja.ipynb` で、元のウインドウと追加ウインドウをまとめて MBAR 解析します。

## Step 1. ライブラリのインポートと環境設定

`05_umbrella_sampling_ja.ipynb` と同じ環境設定です。PLUMED を Python から利用するための環境変数を設定します。

In [ ]:
# 必要に応じてパッケージをインストールしてください
#!pip install joblib
#!pip install plumed

In [ ]:
# ============================================================
# 1. Environment & Imports
# ============================================================

import os
import sys
import glob
import numpy as np
from time import perf_counter
from joblib import Parallel, delayed

# ASE
from ase import units
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.constraints import FixAtoms

# PLUMED wrapper
from ase.calculators.plumed import Plumed

# PFP (Matlantis)
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

# PLUMED Environment Variables (Please modify if necessary to match the path of your environment.)
plumed_path = "/home/jovyan/local/plumed-2.9.0"
os.environ["PLUMED_KERNEL"] = f"{plumed_path}/lib/libplumedKernel.so"
os.environ["PLUMED_TYPESAFE_IGNORE"] = "yes"
sys.path.append(plumed_path)

## Step 2. 計算パラメータの設定

MD の温度・ステップ数は元の計算と揃えます（後段の MBAR で同一温度として扱うため）。
追加サンプリング特有の設定は以下です。

* **KAPPA_STRONG**: 追加ウインドウのバネ定数。元の `2.5 eV` より強くします（デフォルト `10.0 eV`）。
* **cv_restraint_extra**: 追加する拘束中心のリスト。12 Å 付近を 0.1 Å 刻みで細かく配置します。
* **ORIG_US_DIR**: 元のUS出力ディレクトリ。追加ウインドウの初期構造として、最も近い中心の平衡化済み構造を再利用します。
* **OUT_BASE**: 追加ウインドウの出力先。

In [ ]:
# ============================================================
# 2. Parameters & Settings
# ============================================================

# PFP Settings (元の計算と同一にする)
CALC_MODE     = "PBE_PLUS_D3"
METHOD_TYPE   = "PFVM_D3_PFVM"
MODEL_VERSION = "v8.0.0"

# Joblib Settings (並列計算の設定)
N_JOBS  = 10           # 同時に実行する数 (環境に合わせて調整してください)
VERBOSE = 10           # 進捗表示の詳細度
BACKEND = "threading"  # Matlantis利用時は "threading" 推奨

# MD Settings (元の計算と揃える: 温度・タイムステップ・ステップ数)
TEMPERATURE  = 375.0        # Kelvin
TIMESTEP     = 1.0 * units.fs
TOTAL_STEPS  = 50_000       # 各ウィンドウのステップ数
LOG_INTERVAL = 100

# ---- 追加サンプリング固有の設定 ----
# 追加ウインドウのバネ定数 (元は 2.5 eV。強くすることで 12 Å 付近に確実に留める)
KAPPA_STRONG = 10.0         # eV

# 追加する拘束中心 (12 Å 付近を 0.1 Å 刻みで細かく)
cv_restraint_extra = np.round(np.arange(11.6, 12.41, 0.1), 2)

# 引っ張る原子 (PLUMEDは1始まり。ASEの0始まりindex 267 に対応する Cu原子)
PULLED_ATOM = 268
# Cu下層固定の閾値 (z < FIX_THRESH の原子を固定)
FIX_THRESH  = 4.0

# 入出力ディレクトリ
# 元のUS出力 (初期構造の供給元)。output に無ければ assets を使う
ORIG_US_DIR = "./output/05_umbrella_sampling"
if not os.path.isdir(ORIG_US_DIR):
    ORIG_US_DIR = "./assets/05_umbrella_sampling"
    print(f"outputにフォルダが見つからないため、assetsから読み込みます: {ORIG_US_DIR}")
OUT_BASE    = "./output/07_umbrella_sampling_extra"    # 追加ウインドウの出力先 (元を上書きしない)

print(f"KAPPA (strong) : {KAPPA_STRONG} eV  (original 2.5 eV)")
print(f"Extra windows  : {len(cv_restraint_extra)}")
print(f"Values         : {cv_restraint_extra}")
print(f"Output base    : {OUT_BASE}")

## Step 3. 初期構造の供給

追加ウインドウの初期構造は、元の US 計算で得た最も拘束中心が近いウィンドウの平衡化済み構造を再利用します。既に目的の RC 付近まで引き伸ばされた構造なので、
平衡化が早く安定します。

元の US 出力（`output/05_umbrella_sampling/`）が無い場合は、04 で抽出した初期構造（`output/04_initial_structures/`）、それも無ければ `assets/05_umbrella_sampling/` に用意した 0.1 Å 刻みの初期構造を使用するため、このノートブック単独でも実行できます。

In [ ]:
# ============================================================
# 3. Initial structure provider
# ============================================================
# 元のUSウインドウの中心一覧を取得 (05 の出力が無い場合は空リスト)
_orig_centers = sorted(
    float(os.path.basename(d).replace("cv_at_", ""))
    for d in glob.glob(f"{ORIG_US_DIR}/cv_at_*")
)
if _orig_centers:
    print(f"Found {len(_orig_centers)} original windows: "
          f"{_orig_centers[0]:.2f} ... {_orig_centers[-1]:.2f}")
else:
    print(f"No original US windows found in {ORIG_US_DIR}. "
          "Falling back to initial_rc_*.xyz files.")

def find_initial_structure(cv_at):
    """cv_at に最も近い初期構造ファイルのパスを返す。

    優先順位:
      1. 元USウインドウ (ORIG_US_DIR) のうち最も拘束中心が近いウインドウの平衡化済み構造
      2. 04 で抽出した初期構造 (output/04_initial_structures)
      3. assets に用意された初期構造 (assets/05_umbrella_sampling)
    """
    cv_at_str = f"{cv_at:.2f}"

    # 1. 最も近い元USウインドウの平衡化済み構造
    if _orig_centers:
        nearest = min(_orig_centers, key=lambda x: abs(x - cv_at))
        cand = f"{ORIG_US_DIR}/cv_at_{nearest:.2f}/md-dyn-restart.xyz"
        if os.path.exists(cand):
            return cand

    # 2, 3. SMD から抽出した初期構造 (output → assets の順に探す)
    for cand in [f"./output/04_initial_structures/initial_rc_{cv_at_str}.xyz",
                 f"./assets/05_umbrella_sampling/initial_rc_{cv_at_str}.xyz",
                 f"{ORIG_US_DIR}/initial_rc_{cv_at_str}.xyz"]:
        if os.path.exists(cand):
            return cand

    return None

# 供給元の確認
for cv in cv_restraint_extra:
    print(f"cv_at={cv:.2f} <- {find_initial_structure(cv)}")

## Step 4. 追加アンブレラサンプリングを実行する関数

`05_umbrella_sampling_ja.ipynb` の `run_us_window` とほぼ同じですが、以下が異なります。

- バネ定数 `KAPPA` に強い値 `KAPPA_STRONG` を使用
- 初期構造を `find_initial_structure` から取得
- 出力先を `OUT_BASE` 以下に変更（元の結果を上書きしない）
- `COLVAR` の他に、後段の MBAR が読めるよう `KAPPA` 値を `meta.txt` に保存

In [ ]:
# ============================================================
# 4. Parallel Function Definition
# ============================================================
def run_us_window_strong(cv_at):
    """指定した反応座標 cv_at で、強いバネ定数の追加アンブレラサンプリングを実行"""
    s_time = perf_counter()
    cv_at_str = f"{cv_at:.2f}"

    # 1. 出力ディレクトリ
    out_dir = f"{OUT_BASE}/cv_at_{cv_at_str}"
    os.makedirs(out_dir, exist_ok=True)

    # 2. Calculator (並列トラブル防止のため関数内で生成)
    estimator = Estimator(
        calc_mode=CALC_MODE,
        method_type=METHOD_TYPE,
        model_version=MODEL_VERSION,
    )
    calculator = ASECalculator(estimator)

    # 3. Atoms & Constraints
    input_xyz = find_initial_structure(cv_at)
    if input_xyz is None:
        return f"Error: Input structure not found for cv_at={cv_at_str}"
    atoms = read(input_xyz)

    # Cuの1,2層目を固定 (z < FIX_THRESH)
    constraint = FixAtoms(mask=atoms.positions[:, 2] < FIX_THRESH)
    atoms.set_constraint(constraint)

    # 4. PLUMED Settings (強いバネ定数)
    plumed_setting = [
        "UNITS LENGTH=A ENERGY=eV",
        f"pos: POSITION ATOM={PULLED_ATOM}",
        f"restraint-z: RESTRAINT ARG=pos.z KAPPA={KAPPA_STRONG} AT={cv_at_str}",
        f"PRINT STRIDE={LOG_INTERVAL} ARG=pos.z,restraint-z.bias,restraint-z.force2 "
        f"FILE={out_dir}/COLVAR_{cv_at_str}",
        "FLUSH STRIDE=1000",
    ]
    atoms.calc = Plumed(
        calc=calculator,
        input=plumed_setting,
        timestep=TIMESTEP,
        atoms=atoms,
        kT=units.kB * TEMPERATURE,
    )

    # 5. MD Setup
    MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE, force_temp=True)
    Stationary(atoms)
    dyn = Langevin(
        atoms,
        TIMESTEP,
        temperature_K=TEMPERATURE,
        friction=0.002 / units.fs,
        trajectory=f"{out_dir}/md-dyn.traj",
        logfile=f"{out_dir}/md-dyn.log",
        loginterval=LOG_INTERVAL,
        fixcm=False,
    )

    # 6. Run
    try:
        # 後段MBARがウインドウごとのバネ定数を読めるようメタ情報を保存
        with open(f"{out_dir}/meta.txt", "w") as f:
            f.write(f"center {cv_at_str}\n")
            f.write(f"kappa {KAPPA_STRONG}\n")
            f.write(f"temperature {TEMPERATURE}\n")
            f.write(f"init_structure {input_xyz}\n")

        dyn.run(TOTAL_STEPS)
        write(f"{out_dir}/md-dyn-restart.xyz", atoms)
        elapsed = perf_counter() - s_time
        return f"Done: cv_at={cv_at_str} ({elapsed:.1f} sec)"
    except Exception as e:
        return f"Failed: cv_at={cv_at_str} Error: {e}"

## Step 5. 追加アンブレラサンプリングの並列実行

In [ ]:
# ============================================================
# 5. Main Execution
# ============================================================
print(f"Start Parallel Calculation: {len(cv_restraint_extra)} extra windows")
print(f"Settings: n_jobs={N_JOBS}, backend={BACKEND}, KAPPA={KAPPA_STRONG} eV")

results = Parallel(n_jobs=N_JOBS, verbose=VERBOSE, backend=BACKEND)(
    delayed(run_us_window_strong)(cv) for cv in cv_restraint_extra
)

In [ ]:
# 結果の表示
print("\n--- Results ---")
for res in results:
    print(res)

## Step 6. 追加ウインドウのヒストグラムを確認（重なりチェック）

追加した強拘束ウインドウが 12 Å 付近を十分に、かつ隣接ウインドウと重なりながらカバーできているかを確認します。
重なりが不足していれば `cv_restraint_extra` の間隔をさらに詰めるか、`KAPPA_STRONG` を少し弱めてください。

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = "retina"

fig, ax = plt.subplots(figsize=(7, 4))
for cv in cv_restraint_extra:
    f = f"{OUT_BASE}/cv_at_{cv:.2f}/COLVAR_{cv:.2f}"
    if not os.path.exists(f):
        continue
    z = np.loadtxt(f)[1:, 1]
    hist, bins = np.histogram(z, bins=15, density=True)
    ctr = 0.5 * (bins[:-1] + bins[1:])
    ax.plot(ctr, hist, label=f"{cv:.2f}")

ax.set_xlabel("Reaction coordinate z (A)")
ax.set_ylabel("Probability density")
ax.set_title(f"Extra windows (KAPPA={KAPPA_STRONG} eV)")
ax.legend(fontsize=8, ncol=2)
ax.grid(True)
plt.show()

## Next Step

次の [08_mbar_free_energy_with_extra_ja.ipynb](./08_mbar_free_energy_with_extra_ja.ipynb) で、
元のウインドウ（`KAPPA=2.5 eV`）と、この追加ウインドウ（`KAPPA=10.0 eV`）をまとめて MBAR 解析し、
バイアスを除いた自由エネルギープロファイル (PMF) を再計算します。
MBAR はウインドウごとに異なるバネ定数・拘束中心を正しく扱えるため、両者を混在させても問題ありません。